# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  

Set `record_timings = True` to save a per-image timing breakdown CSV alongside the outputs.

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import pandas as pd
from liffile import LifFile

from vascumap import VascuMap
from models import Pix2Pix, load_segmentation_model

W0416 20:18:26.648000 546708 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Helper functions ───────────────────────────────────────────────────────────

def discover_jobs(source_dir, force_mask=None, require_merged=True):
    """Find .lif/.tif/.tiff files and return (source, files, jobs) list."""
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")

    image_files = sorted(
        p for p in source.iterdir()
        if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
    )

    def _mask_mode(p, img_name=""):
        if force_mask is not None:
            return force_mask
        name = (p.name + " " + img_name).lower()
        if "marina" in name and "bead" in name:
            return "light"
        return "dark" if "marina" in name else False

    jobs = []
    for p in image_files:
        if p.suffix.lower() == ".lif":
            try:
                with LifFile(p) as lif:
                    for idx, img in enumerate(lif.images):
                        name = getattr(img, "name", "")
                        if require_merged and "merged" not in name.lower():
                            continue
                        jobs.append((p, idx, _mask_mode(p, name), name))
            except Exception as exc:
                print(f"[SKIP] {p.name}: {exc}")
        else:
            if require_merged and "merged" not in p.name.lower():
                continue
            jobs.append((p, 0, _mask_mode(p), p.stem))

    return source, image_files, jobs


def get_perfect_image_names(perfect_dir):
    """Return a set of image names that have already been perfectly segmented.

    Reads subfolder names from the CATEGORISED/Perfect directory.
    """
    perfect = Path(perfect_dir)
    if not perfect.is_dir():
        print(f"[WARN] Perfect directory not found: {perfect}")
        return set()
    names = {d.name for d in perfect.iterdir() if d.is_dir()}
    print(f"Found {len(names)} perfect images in {perfect}")
    return names


def run_batch(jobs, output_base, device_width_um, brightfield_channel=0,
              save_all_interim=False, model_p2p=None, model_unet=None,
              record_timings=False, start_index=1, skip_names=None):
    """Run all jobs and optionally save a timing CSV.

    Parameters
    ----------
    skip_names : set or None
        Image names to skip (e.g. already-perfect segmentations).
    """
    results, timing_rows = [], []
    if skip_names is None:
        skip_names = set()

    # Ensure the output directory exists
    Path(output_base).mkdir(parents=True, exist_ok=True)

    # Create a directory for failure diagnostics
    failure_diag_dir = Path(output_base) / "FAILED_diagnostics"

    for i, (path, idx, mask_flag, img_name) in enumerate(jobs, start_index):
        tag = f" (LIF #{idx}: {img_name})" if path.suffix.lower() == ".lif" else ""

        # Build the image name the same way the loop body does, so we can
        # check it against the skip list before doing any work.
        if path.suffix.lower() == ".lif":
            safe_name = img_name.replace("/", "_").replace("\\", "_")
            expected_name = f"{path.stem}_{safe_name}_img{idx}"
        else:
            expected_name = path.stem

        if expected_name in skip_names:
            print(f"\n[{i}/{len(jobs)}] {path.name}{tag}  — SKIP (already perfect)")
            results.append((expected_name, "SKIP_PERFECT", ""))
            continue

        print(f"\n{'='*70}\n[{i}/{len(jobs)}] {path.name}{tag}  mask={mask_flag}\n{'='*70}")

        try:
            t0 = time.time()
            vm = VascuMap(
                use_device_segmentation_app=False,
                image_source_path=str(path),
                image_index=idx,
                device_width_um=device_width_um,
                mask_central_region=mask_flag,
                brightfield_channel=brightfield_channel,
                model_p2p=model_p2p,
                model_unet=model_unet,
                failure_output_dir=str(failure_diag_dir),
            )
            vm.image_name = expected_name
            out_dir = Path(output_base) / vm.image_name
            print(f"  Output → {out_dir}")
            vm.pipeline(output_dir=out_dir, save_all_interim=save_all_interim)
            wall = time.time() - t0
            results.append((vm.image_name, "OK", ""))
            if record_timings:
                timing_rows.append({
                    "image_name": vm.image_name, "source_file": path.name,
                    "lif_image_name": img_name,
                    "image_index": idx, "status": "OK",
                    "t_device_seg_s": round(getattr(vm, "_t_device_seg", 0), 1),
                    "t_preprocess_s": round(getattr(vm, "_t_preprocess", 0), 1),
                    "t_inference_s":  round(getattr(vm, "_t_inference", 0), 1),
                    "t_analysis_s":   round(getattr(vm, "_t_analysis", 0), 1),
                    "t_pipeline_s":   round(getattr(vm, "_t_total", 0), 1),
                    "t_job_wall_s":   round(wall, 1),
                })
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((path.name + tag, "FAILED", str(exc)))
            if record_timings:
                timing_rows.append({
                    "image_name": path.name + tag, "source_file": path.name,
                    "lif_image_name": img_name,
                    "image_index": idx, "status": "FAILED",
                })

        if record_timings and timing_rows:
            pd.DataFrame(timing_rows).to_csv(Path(output_base) / "batch_timings.csv", index=False)

    # Summary
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    n_skip = sum(1 for _, s, _ in results if s == "SKIP_PERFECT")
    print(f"\n{'='*70}\nBatch complete: {n_ok}/{len(results)} succeeded, {n_skip} skipped (perfect)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    if record_timings and timing_rows:
        print(f"\nTimings → {Path(output_base) / 'batch_timings.csv'}")
        cols = ["image_name", "lif_image_name", "t_device_seg_s", "t_preprocess_s", "t_inference_s", "t_analysis_s", "t_pipeline_s"]
        print(pd.DataFrame(timing_rows).reindex(columns=cols).to_string(index=False))

    return results

In [3]:
shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)


In [4]:
# ── Configuration (edit here) ──────────────────────────────────────────────────
source_dir  = r"Z:\Maria Cuende\0_Projects\0_Placenta\0_Barrier4PE\3D system\Exp4\Vascumap"
output_base = r"Z:\Bel\Individual_Vascumap_Outputs\Maria_Vascumap"

device_width_um       = 35.0
brightfield_channel   = 0
force_mask            = None     # "dark" / "light" / False / None (auto)
require_merged        = False
save_all_interim      = False
record_timings        = True

In [5]:
# ── Discover jobs + run ────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(source_dir, force_mask, require_merged)

print(f"Source: {source}\nFiles: {len(image_files)}  |  Jobs: {len(jobs)}")
for i, (p, idx, mask, img_name) in enumerate(jobs, 1):
    tag = f" (LIF image {idx}: '{img_name}')" if p.suffix.lower() == ".lif" else ""
    print(f"  {i}. {p.name}{tag}  mask={mask}")

# Skip images already categorised as perfect
perfect_dir = r"Z:\Bel\Individual_Vascumap_Outputs\Maria_Vascumap"
perfect_names = get_perfect_image_names(perfect_dir)

run_batch(jobs, output_base, device_width_um, brightfield_channel,
          save_all_interim, shared_model_p2p, shared_model_unet, record_timings,
          skip_names=perfect_names)

Source: Z:\Maria Cuende\0_Projects\0_Placenta\0_Barrier4PE\3D system\Exp4\Vascumap
Files: 1  |  Jobs: 24
  1. 20260415_Exp4_PE.lif (LIF image 0: 'P1_1')  mask=False
  2. 20260415_Exp4_PE.lif (LIF image 1: 'P2-8')  mask=False
  3. 20260415_Exp4_PE.lif (LIF image 2: 'P1-2')  mask=False
  4. 20260415_Exp4_PE.lif (LIF image 3: 'P1-4')  mask=False
  5. 20260415_Exp4_PE.lif (LIF image 4: 'P5-1')  mask=False
  6. 20260415_Exp4_PE.lif (LIF image 5: 'P5-2')  mask=False
  7. 20260415_Exp4_PE.lif (LIF image 6: 'P5-3')  mask=False
  8. 20260415_Exp4_PE.lif (LIF image 7: 'P5-4')  mask=False
  9. 20260415_Exp4_PE.lif (LIF image 8: 'P5-6')  mask=False
  10. 20260415_Exp4_PE.lif (LIF image 9: 'P2-1')  mask=False
  11. 20260415_Exp4_PE.lif (LIF image 10: 'P2-2')  mask=False
  12. 20260415_Exp4_PE.lif (LIF image 11: 'P2-3')  mask=False
  13. 20260415_Exp4_PE.lif (LIF image 12: 'P2-4')  mask=False
  14. 20260415_Exp4_PE.lif (LIF image 13: 'P2-6')  mask=False
  15. 20260415_Exp4_PE.lif (LIF image 14: 'P3_

Processing chunks: 100%|██████████| 3/3 [00:24<00:00,  8.27s/it]


strong contiguous vote planes 10-15
  ⏱  Stage 2 (Pix2Pix + UNet): 910.0s
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:02<00:00,  2.89s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3
    1475496336.0            1.420256e+09        393614000.0                0.277143           268466.7525

...

Processing chunks: 100%|██████████| 3/3 [00:20<00:00,  6.85s/it]


strong contiguous vote planes 8-17
  ⏱  Stage 2 (Pix2Pix + UNet): 810.2s
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:03<00:00,  3.46s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3
    2029457280.0            1985481988.0       1049723208.0                0.528699           571270.2474

[('20260415_Exp4_PE_P1_1_img0', 'OK', ''),
 ('20260415_Exp4_PE_P2-8_img1', 'OK', ''),
 ('20260415_Exp4_PE_P1-2_img2', 'OK', ''),
 ('20260415_Exp4_PE_P1-4_img3', 'OK', ''),
 ('20260415_Exp4_PE_P5-1_img4', 'OK', ''),
 ('20260415_Exp4_PE_P5-2_img5', 'OK', ''),
 ('20260415_Exp4_PE_P5-3_img6', 'OK', ''),
 ('20260415_Exp4_PE_P5-4_img7', 'OK', ''),
 ('20260415_Exp4_PE_P5-6_img8', 'OK', ''),
 ('20260415_Exp4_PE_P2-1_img9', 'OK', ''),
 ('20260415_Exp4_PE_P2-2_img10', 'OK', ''),
 ('20260415_Exp4_PE_P2-3_img11', 'OK', ''),
 ('20260415_Exp4_PE_P2-4_img12', 'OK', ''),
 ('20260415_Exp4_PE_P2-6_img13', 'OK', ''),
 ('20260415_Exp4_PE_P3_1_img14', 'OK', ''),
 ('20260415_Exp4_PE_P3_4_img15', 'OK', ''),
 ('20260415_Exp4_PE_P3_2_img16', 'OK', ''),
 ('20260415_Exp4_PE_P3_5_img17', 'OK', ''),
 ('20260415_Exp4_PE_P3_7_img18', 'OK', ''),
 ('20260415_Exp4_PE_P4-2_img19', 'OK', ''),
 ('20260415_Exp4_PE_P4-3_img20', 'OK', ''),
 ('20260415_Exp4_PE_P4-4_img21', 'OK', ''),
 ('20260415_Exp4_PE_P4-5_img22', 'OK', '')